In [4]:
from transformers import AutoModelForAudioClassification
import librosa, torch
from tqdm import tqdm
import numpy as np

## Vocals

In [5]:
def chunk_and_run_model(model, sr, mean, std, path, win_s = 3.0, hop_s=0.3):

    # 2. Load + normalize full audio
    wav, _ = librosa.load(path, sr=sr)
    wav = (wav - mean) / (std + 1e-6)

    # 3. Define your window & hop (in seconds)
    W = int(win_s * sr)
    H = int(hop_s * sr)

    # 4. Frame into overlapping chunks
    frames = librosa.util.frame(wav, frame_length=W, hop_length=H).T  # shape=(n_chunks, W)
    
    # 5. Predict on each chunk
    results = []
    for chunk in tqdm(frames):
        x    = torch.from_numpy(chunk).unsqueeze(0)    # (1, W)
        mask = torch.ones(1, W, dtype=torch.long)
        with torch.no_grad():
            out = model(x, mask)                      # (1, 3)
        vec = out.squeeze(0).cpu().numpy()            # [arousal, dominance, valence]                           # start time in seconds
        results.append(vec)
        
    # each point is seperated by hop_s seconds
    return np.array(results)

### vocal - regression 

`{0: 'arousal', 1: 'dominance', 2: 'valence'}`

In [6]:
#load model
model = AutoModelForAudioClassification.from_pretrained("3loi/SER-Odyssey-Baseline-WavLM-Multi-Attributes", trust_remote_code=True).eval()

#get mean/std
sr   = model.config.sampling_rate
mean = model.config.mean
std  = model.config.std

# in seconds
win_s = 3.0; hop_s = 0.3

results = chunk_and_run_model(model, sr, mean, std, "vocals.wav", win_s, hop_s)

  0%|          | 0/1581 [00:00<?, ?it/s]/home/nuttidalab/miniconda3/envs/movie/lib/python3.12/site-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
100%|██████████| 1581/1581 [02:50<00:00,  9.27it/s]


In [8]:
# results is now a list of (time, [a,d,v]) tuples
for vec in results[:5]:
    print(f"arousal={vec[0]:.2f}, domin={vec[1]:.2f}, valence={vec[2]:.2f}")
#{0: 'arousal', 1: 'dominance', 2: 'valence'}
#tensor([[0.3670, 0.4553, 0.4240]])

arousal=0.56, domin=0.57, valence=0.52
arousal=0.37, domin=0.39, valence=0.45
arousal=0.34, domin=0.35, valence=0.42
arousal=0.33, domin=0.34, valence=0.42
arousal=0.43, domin=0.45, valence=0.46


In [10]:
results.shape

(1581, 3)

In [ ]:
# np.save("vocal_attributes.npy", results)

### vocal - classification 



In [15]:
#load model
model = AutoModelForAudioClassification.from_pretrained("3loi/SER-Odyssey-Baseline-WavLM-Categorical-Attributes", trust_remote_code=True).eval()

#get mean/std
sr   = model.config.sampling_rate
mean = model.config.mean
std  = model.config.std

# in seconds
win_s = 3.0; hop_s = 0.3

results = chunk_and_run_model(model, sr, mean, std, "vocals.wav", win_s, hop_s)

config.json:   0%|          | 0.00/750 [00:00<?, ?B/s]

pipeline_utils.py:   0%|          | 0.00/5.55k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/3loi/SER-Odyssey-Baseline-WavLM-Categorical-Attributes:
- pipeline_utils.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

  0%|          | 0/1581 [00:00<?, ?it/s]/home/nuttidalab/miniconda3/envs/movie/lib/python3.12/site-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
100%|██████████| 1581/1581 [03:03<00:00,  8.59it/s]


In [25]:
# convert logits to probabilities
probs = torch.nn.functional.softmax(torch.from_numpy(results), dim=-1).numpy()
results.shape, probs.shape

((1581, 8), (1581, 8))

In [27]:
model.config.id2label

{0: 'Angry',
 1: 'Sad',
 2: 'Happy',
 3: 'Surprise',
 4: 'Fear',
 5: 'Disgust',
 6: 'Contempt',
 7: 'Neutral'}

In [30]:
import pandas as pd

In [ ]:
# pd.DataFrame(probs, columns=[model.config.id2label[i] for i in range(len(model.config.id2label))], index=np.arange(len(probs))*hop_s).to_parquet("vocal_emos.parquet")

## Instrumental (music2emo)